# RBAC + ABAC baselines (rule-based, no training)

**CPU only** -- plain pandas/rule evaluation, no model training, no GPU benefit.

Two baselines, both deterministic (nothing learned, nothing to train):

- **RBAC**: role assigned once per identity type, fixed permission, no behavioral awareness at all -- the deliberately least-adaptive comparison point. Quantifies what a non-adaptive baseline misses.
- **ABAC**, in TWO variants: **single-authority** (CPO trust alone -- what an ABAC deployment looks like without a multi-authority architecture) and **multi-authority** (CPO + eMSP + EV trust, combined weakest-link). Comparing the two directly measures what single-authority ABAC structurally cannot see -- an honest-looking CPO paired with a compromised eMSP or a Sybil EV -- which is the empirical case for needing a multi-authority dataset/system, not an assumption the benchmark takes for granted.

Both ABAC variants reuse `decision_engine.py`'s existing 3-way PERMIT / CONDITIONAL-PERMIT / DENY decision (same theta_high=0.75/theta_low=0.4 thresholds TrustMesh itself uses), fed by NAIVE (unfused, no subjective logic) per-authority trust -- so this stays an independent baseline, not TrustMesh's own policy layer wearing a different hat.

**Now variant-aware, AND fixes a found bug (read this if you ran an earlier version of this notebook):** `naive_trust.py`'s causal running statistics (`naive_cpo_trust`, `naive_ev_trust`) used to sort the ENTIRE passed-in dataframe by `connection_time` and keep one running-sum dict per `cpo_id`/`ev_id`, with no awareness of which world a row came from. World 0 and world 1 share the SAME real ACN-Data timestamps (only synthetic fields differ), so a tied timestamp let world 1's running trust for a given CPO silently continue from wherever world 0 left off instead of starting cold -- **this was already true for the original 2-world `main` dataset**, not something the 7-variant work introduced, just made more severe (now 14 (variant, world) groups instead of 2 plain worlds sharing one timeline). Fixed by grouping the causal computation by `(variant, world_id)` before computing each entity's running statistic -- see `naive_trust.py`'s updated BUG LOG for the full account. This also means `cpo_trust_scale` is now a PER-VARIANT calibration (`noisy_benign`'s honest population is deliberately noisier than `main`'s, so it gets its own, looser scale -- see `compute_cpo_trust_scale`'s docstring).

**Requires:** `v2g_benchmark_multi_world.csv` and the day cutoff from `03_prepare_baseline_labels_splits.ipynb` (recomputed here identically, not reloaded, since both RBAC and ABAC need the FULL chronological sequence -- see `naive_trust.py`'s docstring for why).

## Step 1: write the shared decision layer + ground truth/split modules

`decision_engine.py` is the EXISTING, already-validated TrustMesh decision layer (Decision enum, `trust_weighted_decision`, `evaluate_access_policy`) -- reused as-is, not reimplemented, so every baseline shares the identical decision granularity.

In [ ]:
%%writefile decision_engine.py
"""
decision_engine.py
-------------------
Implements Algorithm 4 from the TrustMesh framework doc (Section 8):
the trust-weighted three-way access decision (PERMIT / CONDITIONAL-PERMIT / DENY).

This is what turns a trust score into an actual access-control outcome,
which is the concrete instantiation of Gap 4.3.1 (fuzzy trust feeding
MA-ABAC decisions directly, not sitting beside them).
"""

from enum import Enum


class Decision(Enum):
    PERMIT = "PERMIT"
    CONDITIONAL_PERMIT = "CONDITIONAL-PERMIT"
    DENY = "DENY"


def trust_weighted_decision(base_confidence: float, effective_trust: float,
                             theta_high: float = 0.75, theta_low: float = 0.4) -> Decision:
    """
    effective_confidence = base_confidence * effective_trust   (framework doc Section 8)

    base_confidence: how strongly the attribute itself is asserted (e.g., a
    validly-signed certificate = 1.0; a partially-expired one might be lower).
    effective_trust: E[opinion] = b + a*u for the issuing authority.
    """
    effective_confidence = base_confidence * effective_trust
    if effective_confidence >= theta_high:
        return Decision.PERMIT
    elif effective_confidence >= theta_low:
        return Decision.CONDITIONAL_PERMIT
    else:
        return Decision.DENY


# ---------------------------------------------------------------------------
# Multi-attribute ABAC policy (combines authority_trust with OTHER
# attributes via rules, not just a single trust threshold -- this is what
# makes it attribute-based access control rather than trust-gating alone).
# ---------------------------------------------------------------------------

ROAMING_TRUST_PENALTY = 0.10  # roaming sessions apply a direct penalty to the
                                # CPO's trust value -- a real, common pattern
                                # (foreign/roaming access needs stronger
                                # assurance than home-network access)


def evaluate_access_policy(cpo_trust: float, emsp_trust: float, certificate_valid: bool,
                             is_roaming: bool, is_peak_hour: bool,
                             theta_high: float = 0.75, theta_low: float = 0.4,
                             roaming_penalty: float = ROAMING_TRUST_PENALTY) -> Decision:
    """
    GENUINE multi-authority ABAC policy: the decision is a function of TWO
    independently-computed authorities' trust scores, not one authority's
    trust plus trust-blind attribute checks.

      - cpo_trust:  TrustMesh's trust in the CPO (billing honesty)
      - emsp_trust: the hub-mediated, EFFECTIVE confidence in the eMSP
                    (certificate-issuance track record), already discounted
                    by how much the relying CPO trusts the hub's reporting
                    (see trust_pipeline.effective_emsp_confidence) -- this is
                    the actual cross-domain fusion mechanism validated in
                    the cold-start experiment, not a flat heuristic.
      - certificate_valid: a per-session, OBJECTIVE fact (hard constraint,
                    e.g. a CRL check) -- NOT subject to either authority's
                    trust score, since it is independently verifiable.

    Combination rule: MIN (weakest-link / AND-conjunction) of the two
    authority trust scores -- both the charging network's billing honesty
    AND the driver's home-network credential issuer must independently
    clear the bar; one being excellent cannot compensate for the other
    being poor. This is what "multi-authority" means operationally: neither
    score alone determines the outcome.
    """
    if not certificate_valid:
        return Decision.DENY  # objective fact, not subject to either authority's trust

    adjusted_cpo_trust = cpo_trust - (roaming_penalty if is_roaming else 0.0)
    combined_trust = min(adjusted_cpo_trust, emsp_trust)

    decision = trust_weighted_decision(1.0, combined_trust, theta_high, theta_low)
    if is_peak_hour and decision == Decision.PERMIT:
        decision = Decision.CONDITIONAL_PERMIT  # environment override, downgrade only

    return decision


In [ ]:
%%writefile benchmark_labels.py
"""
benchmark_labels.py
----------------------
Ground-truth labels shared by EVERY baseline (RBAC, ABAC, ML, DL, RL,
graph-structural) and by TrustMesh itself, so all of them are scored
against the exact same target.

GROUNDING LEDGER:
  - gt_cpo_malicious: directly from injection.py's authority_malicious
    column (1 iff cpo_id == the injected malicious CPO AND day_idx >=
    onset_day). Already a column in v2g_benchmark_multi_world.csv --
    not recomputed here, just carried through.
  - gt_emsp_malicious: ONSET-GATING FIX -- this USED to be a pure, permanent
    function of emsp_id == MALICIOUS_EMSP (identity_assignment.py's eMSP-C),
    with zero temporal variation: the transformer-baseline entity-attention
    proof of concept's own entity-only ablation confirmed this made the axis
    structurally unlearnable from behavior for ANY entity-aggregation model
    (mean- or attention-pooling), since there was never a honest-vs-malicious
    contrast in eMSP-C's own history to learn from, only its name. Now reads
    multi_world_pipeline.py/feature_engineering.py's persisted
    emsp_malicious_eligible column directly (1 iff emsp_id == MALICIOUS_EMSP
    AND day_idx >= emsp_onset_day) -- the same "identity AND onset" shape
    authority_malicious already has for CPO, so the two axes are symmetric.
    A combined CSV that predates this fix won't have that column;
    add_ground_truth_labels raises a clear error rather than silently
    falling back to the old identity-only definition. Independent threat
    axis from CPO billing fraud (independent, staggered onset day).
  - gt_ev_sybil: directly from enriched_features.py's ev_sybil column.
    Independent threat axis from both of the above.
  - risk_factor_count / composite_risk_score: NOT data fields -- these
    are EVALUATION-TARGET design choices made HERE, not anything the
    generation pipeline produces or implies. Said plainly so they can
    be cited precisely in the paper rather than mistaken for something
    collected "as-is": risk_factor_count is a simple count (0-3) of how
    many of the three independent threat axes are active for a given
    session; composite_risk_score is that count turned into [0,1] under
    an EQUAL-WEIGHTING assumption across axes (AXIS_WEIGHTS below) --
    not a claim that CPO billing fraud, eMSP credential issues, and EV
    Sybil attacks carry equal real-world severity.

WHY NOT A BAKED-IN "DECISION" COLUMN HERE: every baseline's own
continuous score gets passed through decision_engine.trust_weighted_decision()
at evaluation time (04_performance_analysis.ipynb), not pre-computed here.
That keeps the PERMIT / CONDITIONAL-PERMIT / DENY granularity a property
of each model's score, not of the ground truth -- which is what lets the
benchmark show every baseline (not just TrustMesh) supporting that same
granularity, and lets a user push to a finer granularity later just by
changing threshold cut points, with nothing in this module to revisit.
"""

import numpy as np
import pandas as pd

MALICIOUS_EMSP = "eMSP-C"  # mirrors identity_assignment.py; duplicated here
                            # deliberately so this module has no import-time
                            # dependency on trustmesh_pkg's package layout

AXIS_WEIGHTS = {"cpo": 1.0, "emsp": 1.0, "ev": 1.0}  # documented equal-weighting
                                                        # assumption, see module docstring


def add_ground_truth_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["gt_cpo_malicious"] = df["authority_malicious"].astype(int)

    if "emsp_malicious_eligible" not in df.columns:
        raise KeyError(
            "Column 'emsp_malicious_eligible' not found. This combined CSV predates "
            "the eMSP onset-gating fix (see multi_world_pipeline.py / "
            "feature_engineering.py, and this module's GROUNDING LEDGER above) -- "
            "regenerate the dataset (02_generate_v2g_benchmark.ipynb) before running "
            "this baseline. Refusing to silently fall back to the old, structurally-"
            "unlearnable identity-only definition (emsp_id == MALICIOUS_EMSP)."
        )
    df["gt_emsp_malicious"] = df["emsp_malicious_eligible"].astype(int)
    df["gt_ev_sybil"] = df["ev_sybil"].astype(int)

    df["risk_factor_count"] = (
        df["gt_cpo_malicious"] + df["gt_emsp_malicious"] + df["gt_ev_sybil"]
    )
    total_weight = sum(AXIS_WEIGHTS.values())
    df["composite_risk_score"] = (
        AXIS_WEIGHTS["cpo"] * df["gt_cpo_malicious"]
        + AXIS_WEIGHTS["emsp"] * df["gt_emsp_malicious"]
        + AXIS_WEIGHTS["ev"] * df["gt_ev_sybil"]
    ) / total_weight
    return df


def summarize_labels(df: pd.DataFrame) -> None:
    print("risk_factor_count distribution (0 = clean, 3 = all three threat axes active):")
    print(df["risk_factor_count"].value_counts().sort_index())
    print(f"\nAny risk factor present: {(df['risk_factor_count'] > 0).mean():.4f}")
    for axis in ["gt_cpo_malicious", "gt_emsp_malicious", "gt_ev_sybil"]:
        print(f"  {axis} rate: {df[axis].mean():.4f}")
    print(f"\ncomposite_risk_score describe:\n{df['composite_risk_score'].describe()}")


if __name__ == "__main__":
    import os
    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = add_ground_truth_labels(df)
    summarize_labels(df)


In [ ]:
%%writefile temporal_split.py
"""
temporal_split.py
--------------------
Train/test split for baseline evaluation: temporal within each world,
not random and not by-world.

WHY TEMPORAL: day_idx is REAL (derived from actual ACN-Data connection
timestamps, see raw_preprocess.py) and IDENTICAL in range across both
worlds -- only the synthetic identity/fraud fields differ by world, not
the real session timing. injection.py's attacks are keyed on day_idx
(onset_day), so a model that has only ever seen days before the cutoff
has a genuinely harder, more realistic job on the test split than a
model trained on a random shuffle that leaks future-day sessions into
training. This is the right split for a benchmark meant to measure "how
well/fast does each baseline detect or adapt to a trust problem," not
just "can it memorize this distribution."

Applied with the SAME day cutoff across all worlds (computed once, on
the full combined frame), since day_idx's distribution is shared real
data, not an independent per-world draw.

ONE DISCLOSED CAVEAT: enriched_features.add_baseline_charge_duration's
station+hour empirical-median fallback was computed over the WHOLE
dataset (train+test combined) when the benchmark CSV was generated, not
separately per split -- a minor leakage source for that one contextual
feature column specifically. Harmless for anything that doesn't use
baseline_charge_duration_h as a predictor of risk (it isn't one), but
worth stating plainly rather than silently ignoring, consistent with
this project's disclosure discipline.
"""

import numpy as np
import pandas as pd

TRAIN_FRACTION_OF_DAYS = 0.7  # documented split point, not tuned


def compute_day_cutoff(df: pd.DataFrame, train_fraction: float = TRAIN_FRACTION_OF_DAYS) -> int:
    days = np.sort(df["day_idx"].unique())
    cutoff_idx = int(len(days) * train_fraction)
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both splits non-empty
    return int(days[cutoff_idx])


def temporal_train_test_split(df: pd.DataFrame, day_cutoff: int = None):
    if day_cutoff is None:
        day_cutoff = compute_day_cutoff(df)
    train = df[df["day_idx"] < day_cutoff].copy()
    test = df[df["day_idx"] >= day_cutoff].copy()
    print(f"[split] day_cutoff={day_cutoff}  train={len(train)} rows "
          f"(days {train['day_idx'].min()}-{train['day_idx'].max()})  "
          f"test={len(test)} rows (days {test['day_idx'].min()}-{test['day_idx'].max()})")
    for name, part in [("train", train), ("test", test)]:
        rate = (part["risk_factor_count"] > 0).mean() if "risk_factor_count" in part.columns else float("nan")
        print(f"  {name}: any-risk-factor rate = {rate:.4f}")
    return train, test, day_cutoff


if __name__ == "__main__":
    import os
    import benchmark_labels

    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = benchmark_labels.add_ground_truth_labels(df)
    train, test, cutoff = temporal_train_test_split(df)
    print(f"\nWorld breakdown -- train:\n{train['world_id'].value_counts()}")
    print(f"World breakdown -- test:\n{test['world_id'].value_counts()}")


## Step 2: write the naive-trust, RBAC, and ABAC modules

`naive_trust.py` now groups every causal running statistic by `(variant, world_id)` -- see its module docstring's BUG LOG entry above.

In [ ]:
%%writefile naive_trust.py
"""
naive_trust.py
----------------
Naive (unfused) per-authority trust trajectories for the ABAC baseline --
deliberately NOT using TrustMesh's own Algorithm 1 (fuzzy engine,
fuzzy_trust.py) or Algorithm 2 (subjective-logic propagation,
subjective_logic.py), so the ABAC baseline stays independent of
TrustMesh rather than secretly running it underneath (TrustMesh feeding
its OWN trust into an ABAC policy layer is a separate, already-published
result -- see trustmesh_pkg/run_experiments.py's Experiment 6). Each
function here is a plain, CAUSAL (strictly-prior-session, no look-ahead)
running statistic -- nothing in this module looks at a row's own future.

GROUNDING LEDGER:
  - naive_cpo_trust: a LINEAR (not fuzzy-membership) rescaling of the
    causal running mean of billing_deviation for that CPO, against a
    SCALE calibrated from the honest population (same calibrate-from-
    honest-population discipline as fuzzy_trust.py, just linear instead
    of triangular-membership, and deliberately calibrated from the
    TRAIN split only -- see compute_cpo_trust_scale -- so the test split
    never influences the calibration constant).
  - naive_emsp_trust: 1 - previous_flag_rate. previous_flag_rate ALREADY
    EXISTS as a column (enriched_features.py) -- reused directly, not
    recomputed: it's already exactly this, a causal, prior-only running
    dispute rate per eMSP.
  - naive_ev_trust: NEW. A causal running "session-share" statistic per
    EV -- the fraction of all sessions seen so far that belong to this
    EV, compared against the uniform-usage expectation
    (1 / N_SYNTHETIC_EVS). A Sybil EV's defining signature (see
    enriched_features.inject_ev_sybil) is reusing the SAME identity far
    more than organic usage would -- this is a direct, simple read on
    exactly that, with no graph structure needed (the graph-structural
    baseline gets that machinery; this is deliberately the plainer,
    non-structural comparison point available to a policy engine that
    has no graph layer at all).

BUG LOG (found while wiring up the 7-build variant suite, confirmed
PRE-EXISTING -- present even for the original single-build, 2-world
"main" dataset, not something the variant work introduced): both
add_naive_cpo_trust and add_naive_ev_trust sort the ENTIRE passed-in
dataframe by connection_time and keep one running_sum/running_n dict
per cpo_id/ev_id, with no awareness of which world or variant a row
belongs to. World 0 and world 1 (and, now, all 7 variants) are built
from the SAME real ACN-Data connection_time values -- only the
synthetic identity/fraud/feature draws differ -- so two rows that are
genuinely independent synthetic universes can carry the IDENTICAL
timestamp. Sorting the union by connection_time interleaves them, and
a tied timestamp breaks ties by original row order (world 0 rows were
concatenated before world 1's in multi_world_pipeline.py), so world
1's running trust for "CPO-B" silently started from wherever world 0's
running sum happened to be instead of a clean cold start -- the same
leak, now across 14 (variant, world) groups instead of 2 plain worlds.
Fixed by grouping the causal computation by (variant, world_id) before
running each entity's running statistic, falling back to no grouping
when those columns aren't present (e.g. a single-world/single-variant
subset passed in directly) so existing single-build callers are
unaffected. add_naive_emsp_trust is NOT affected: it just reads
previous_flag_rate, which enriched_features.py already computes
per-world at generation time, before any combining.
"""

import numpy as np
import pandas as pd

N_SYNTHETIC_EVS = 1000  # mirrors identity_assignment.py
EV_SHARE_SUSPICION_MULTIPLE = 5.0  # documented assumption: an EV using more
                                     # than 5x its uniform-expected share of
                                     # sessions seen so far is treated as
                                     # fully untrustworthy (naive_ev_trust ->
                                     # 0); linear in between, 0 share -> 1.0

DEFAULT_GROUP_COLS = ("variant", "world_id")  # each (variant, world_id) pair is
# an independent synthetic universe sharing the same real connection_time
# range as every other one -- causal running statistics must never cross
# this boundary (see BUG LOG above).


def _present_group_cols(df: pd.DataFrame, group_cols) -> list:
    return [c for c in group_cols if c in df.columns]


def _iter_groups(df: pd.DataFrame, group_cols):
    """Yields (key, sub_df) pairs to compute a causal running statistic
    independently within. Falls back to the whole frame as one group when
    none of group_cols are present, so single-world/single-variant callers
    (or a caller that already sliced to one world) see no behavior change."""
    cols = _present_group_cols(df, group_cols)
    if not cols:
        yield None, df
        return
    for key, g in df.groupby(cols, sort=False):
        yield key, g


def compute_cpo_trust_scale(train_df: pd.DataFrame, malicious_cpo: str,
                               quantile: float = 0.999, per_variant: bool = True):
    """Calibration constant for add_naive_cpo_trust, derived ONLY from the
    TRAIN split's apparently-honest rows (cpo_id != malicious_cpo) -- never
    from test, so the one calibrated number in this module can't leak
    test-period information into the baseline.

    When train_df spans more than one variant and per_variant=True (the
    default), returns a DICT {variant_name: scale} instead of a single
    float: noisy_benign's honest population is deliberately noisier than
    main's, so calibrating every variant off one pooled quantile would
    judge noisy_benign's honest sessions against a scale that's too tight
    (leaked from the other, quieter variants) and make them look less
    trustworthy than main's honest sessions for no behavioral reason.
    Pass per_variant=False, or call on a single-variant frame, to get the
    original single-float behavior back."""
    if per_variant and "variant" in train_df.columns:
        out = {}
        for vname, g in train_df.groupby("variant", sort=False):
            honest = g.loc[g["cpo_id"] != malicious_cpo, "billing_deviation"]
            s = float(honest.quantile(quantile))
            out[vname] = s if s > 1e-9 else 1e-6
        return out
    honest = train_df.loc[train_df["cpo_id"] != malicious_cpo, "billing_deviation"]
    scale = float(honest.quantile(quantile))
    return scale if scale > 1e-9 else 1e-6


def add_naive_cpo_trust(df: pd.DataFrame, scale, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """df should be the FULL chronological sequence (all worlds/rows you want
    trajectories over) -- causal, so safe to run across train+test combined;
    only `scale` (from compute_cpo_trust_scale) must come from train alone.

    `scale` is either a single float (applied to every row) or a dict
    {variant_name: float} (see compute_cpo_trust_scale) -- resolved per
    group below. The running statistic itself is computed independently
    within each (variant, world_id) group (see module BUG LOG): grouping
    first, then sorting each group by connection_time, keeps every group's
    running sums starting from a clean cold start instead of leaking into
    each other through shared/tied real timestamps."""
    df = df.copy()
    trust = pd.Series(np.nan, index=df.index, dtype=float)
    for key, g in _iter_groups(df, group_cols):
        if isinstance(scale, dict):
            variant_cols = _present_group_cols(df, ("variant",))
            this_scale = scale[g["variant"].iloc[0]] if variant_cols else next(iter(scale.values()))
        else:
            this_scale = scale

        order = g.sort_values("connection_time").index
        bd = g.loc[order, "billing_deviation"].to_numpy()
        cpo = g.loc[order, "cpo_id"].to_numpy()

        running_sum: dict = {}
        running_n: dict = {}
        local = np.empty(len(g))
        for i, c in enumerate(cpo):
            n = running_n.get(c, 0)
            running_mean = (running_sum.get(c, 0.0) / n) if n > 0 else 0.0  # cold start: assume trustworthy
            local[i] = float(np.clip(1.0 - running_mean / this_scale, 0.0, 1.0))
            running_sum[c] = running_sum.get(c, 0.0) + bd[i]
            running_n[c] = n + 1
        trust.loc[order] = local

    df["naive_cpo_trust"] = trust
    return df


def add_naive_emsp_trust(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["naive_emsp_trust"] = 1.0 - df["previous_flag_rate"]
    return df


def add_naive_ev_trust(df: pd.DataFrame, n_evs: int = N_SYNTHETIC_EVS,
                          suspicion_multiple: float = EV_SHARE_SUSPICION_MULTIPLE,
                          group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """See module BUG LOG: the running EV session-share statistic is
    computed independently within each (variant, world_id) group, not
    across the full passed-in frame, for the same reason as
    add_naive_cpo_trust above."""
    df = df.copy()
    trust = pd.Series(np.nan, index=df.index, dtype=float)

    uniform_share = 1.0 / n_evs
    suspicious_share = uniform_share * suspicion_multiple

    for key, g in _iter_groups(df, group_cols):
        order = g.sort_values("connection_time").index
        ev = g.loc[order, "ev_id"].to_numpy()

        running_n: dict = {}
        total_seen = 0
        local = np.empty(len(g))
        for i, e in enumerate(ev):
            this_ev_n = running_n.get(e, 0)
            share_so_far = (this_ev_n / total_seen) if total_seen > 0 else 0.0  # cold start: trustworthy
            local[i] = float(np.clip(1.0 - share_so_far / suspicious_share, 0.0, 1.0))
            running_n[e] = this_ev_n + 1
            total_seen += 1
        trust.loc[order] = local

    df["naive_ev_trust"] = trust
    return df


def add_all_naive_trust(df: pd.DataFrame, cpo_trust_scale, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    df = add_naive_cpo_trust(df, cpo_trust_scale, group_cols=group_cols)
    df = add_naive_emsp_trust(df)
    df = add_naive_ev_trust(df, group_cols=group_cols)
    return df


In [ ]:
%%writefile rbac_baseline.py
"""
rbac_baseline.py
-------------------
RBAC baseline: the deliberately LEAST adaptive comparison point. A role
is assigned ONCE per identity type (CPO / eMSP / EV), carrying a FIXED
permission that never updates from observed behavior -- this is what
classical Role-Based Access Control actually is: permission follows the
role, not a running trust estimate. The only thing that can override the
role's default PERMIT is an OBJECTIVE, per-session hard fact
(certificate_valid), mirroring decision_engine.evaluate_access_policy's
own treatment of that same fact as a hard gate independent of trust.

Structurally, this baseline CANNOT detect CPO billing fraud, eMSP
dispute history, or EV Sybil reuse -- none of those are visible to a
role lookup, by construction. That is the point of including it: it
quantifies what a non-adaptive baseline misses, which is the gap every
other baseline (ABAC, ML, DL, RL, graph-structural, TrustMesh) exists to
close.
"""

import pandas as pd
from decision_engine import Decision


def rbac_decide(certificate_valid: bool) -> Decision:
    return Decision.DENY if not certificate_valid else Decision.PERMIT


def add_rbac_decision(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["rbac_decision"] = df["certificate_valid"].apply(
        lambda v: rbac_decide(v).value
    )
    return df


def summarize_rbac_blindness(df: pd.DataFrame) -> None:
    """Quantifies the structural limitation: of sessions with a REAL risk
    factor present (CPO fraud, eMSP dispute-prone, or EV Sybil), what
    fraction does RBAC still PERMIT, purely because it has no behavioral
    signal to act on?"""
    risky = df[df["risk_factor_count"] > 0]
    print(f"Sessions with >=1 real risk factor: {len(risky)}/{len(df)}")
    if len(risky):
        dist = risky["rbac_decision"].value_counts(normalize=True) * 100
        print("RBAC decision distribution on those sessions (expect mostly PERMIT -- "
              "RBAC has no way to see the risk):")
        print(dist.round(1).to_dict())
        permit_rate = (risky["rbac_decision"] == Decision.PERMIT.value).mean()
        print(f"\nRBAC PERMITs {100 * permit_rate:.1f}% of genuinely risky sessions "
              f"outright -- the gap every other baseline exists to close.")


In [ ]:
%%writefile abac_baseline.py
"""
abac_baseline.py
-------------------
Two ABAC variants, both reusing decision_engine's Decision enum and
3-way PERMIT / CONDITIONAL-PERMIT / DENY thresholds (theta_high=0.75,
theta_low=0.4) so they sit on the same decision scale as every other
baseline and as TrustMesh itself -- not reduced to binary grant/deny.

  - single_authority_decision: the classical single-relying-party ABAC
    policy -- the decision depends on ONE authority's trust (the CPO's)
    plus the objective certificate_valid hard gate. This is what an
    ABAC deployment looks like WITHOUT a multi-authority trust
    architecture: whatever the CPO's own trust score says, goes, and
    nothing else is consulted.
  - multi_authority_decision: combines THREE independently-computed
    authority trust scores (CPO billing honesty, eMSP certificate-
    issuance track record, EV identity legitimacy) via MIN
    (weakest-link) -- extends decision_engine.evaluate_access_policy's
    existing 2-authority (CPO+eMSP) design to the 3rd axis (EV Sybil)
    this richer benchmark adds. One bad authority drags the decision
    down regardless of how good the other two look.

Both are fed NAIVE (unfused) per-authority trust from naive_trust.py --
deliberately, so this ABAC baseline stays independent of TrustMesh's own
Algorithm 1/2 rather than secretly running TrustMesh underneath. ABAC
here is benchmarked AS AN ALTERNATIVE to TrustMesh, not as TrustMesh's
own policy layer (that integration is Experiment 6 in
trustmesh_pkg/run_experiments.py -- a separate, already-published
result).

THE VALIDATION THIS BUYS: comparing single_authority_decision against
multi_authority_decision on the SAME sessions directly measures what a
single-authority ABAC deployment misses -- specifically, sessions with a
trustworthy-looking CPO but a compromised eMSP or a Sybil EV, which
single-authority ABAC structurally cannot see (it only ever looks at the
CPO). If multi-authority materially changes the decision on a
meaningful fraction of exactly those sessions, that is direct empirical
evidence for needing a multi-authority dataset/system in the first
place, not an assumption the benchmark takes for granted.
"""

import numpy as np
import pandas as pd
from decision_engine import Decision, trust_weighted_decision, ROAMING_TRUST_PENALTY


def single_authority_decision(cpo_trust: float, certificate_valid: bool,
                                 theta_high: float = 0.75, theta_low: float = 0.4) -> Decision:
    if not certificate_valid:
        return Decision.DENY
    return trust_weighted_decision(1.0, cpo_trust, theta_high, theta_low)


def multi_authority_decision(cpo_trust: float, emsp_trust: float, ev_trust: float,
                                certificate_valid: bool, is_roaming: bool, is_peak_hour: bool,
                                theta_high: float = 0.75, theta_low: float = 0.4,
                                roaming_penalty: float = ROAMING_TRUST_PENALTY) -> Decision:
    if not certificate_valid:
        return Decision.DENY

    adjusted_cpo_trust = cpo_trust - (roaming_penalty if is_roaming else 0.0)
    combined_trust = min(adjusted_cpo_trust, emsp_trust, ev_trust)

    decision = trust_weighted_decision(1.0, combined_trust, theta_high, theta_low)
    if is_peak_hour and decision == Decision.PERMIT:
        decision = Decision.CONDITIONAL_PERMIT  # environment override, downgrade only
    return decision


def add_abac_decisions(df: pd.DataFrame) -> pd.DataFrame:
    """Requires naive_cpo_trust, naive_emsp_trust, naive_ev_trust,
    certificate_valid, is_roaming, is_peak_hour already present (see
    naive_trust.add_all_naive_trust)."""
    df = df.copy()
    df["abac_single_decision"] = [
        single_authority_decision(r.naive_cpo_trust, r.certificate_valid).value
        for r in df.itertuples()
    ]
    df["abac_multi_decision"] = [
        multi_authority_decision(r.naive_cpo_trust, r.naive_emsp_trust, r.naive_ev_trust,
                                   r.certificate_valid, r.is_roaming, r.is_peak_hour).value
        for r in df.itertuples()
    ]
    return df


def summarize_single_vs_multi(df: pd.DataFrame) -> None:
    differs = df["abac_single_decision"] != df["abac_multi_decision"]
    print(f"Single-authority vs multi-authority ABAC differ on "
          f"{differs.sum()}/{len(df)} sessions ({100 * differs.mean():.2f}%)")

    # The direct validation the dataset's multi-authority framing needs:
    # sessions where the CPO looks fine but another axis is actually
    # malicious -- single-authority ABAC structurally cannot see these,
    # since it never looks past the CPO.
    blind_spot = df[(df["gt_cpo_malicious"] == 0) &
                     ((df["gt_emsp_malicious"] == 1) | (df["gt_ev_sybil"] == 1))]
    print(f"\nSessions with an honest-looking CPO but a malicious eMSP or Sybil EV: "
          f"{len(blind_spot)}")
    if len(blind_spot):
        print("  Single-authority ABAC decision distribution for these "
              "(should NOT be mostly PERMIT if the risk is real):")
        print("  ", blind_spot["abac_single_decision"].value_counts().to_dict())
        print("  Multi-authority ABAC decision distribution for the SAME sessions:")
        print("  ", blind_spot["abac_multi_decision"].value_counts().to_dict())
        not_denied_single = (blind_spot["abac_single_decision"] != Decision.DENY.value).mean()
        denied_multi = (blind_spot["abac_multi_decision"] == Decision.DENY.value).mean()
        print(f"\n  Single-authority ABAC does NOT deny (grants PERMIT or CONDITIONAL-PERMIT "
              f"to) {100 * not_denied_single:.1f}% of these blind-spot sessions; "
              f"multi-authority ABAC denies {100 * denied_multi:.1f}% of the SAME sessions "
              f"-- the concrete case for a multi-authority system, not an assumption.")


## Step 3: run both baselines on the full chronological sequence, reported per variant

Naive trust is a CAUSAL running statistic (see `naive_trust.py`), so it's computed across the full train+test timeline at once -- that's what makes a test-period row correctly reflect everything learned before it, train and earlier test rows alike, instead of resetting to cold-start at the split boundary. The one calibrated number per variant (`cpo_trust_scale`) is still derived from TRAIN-only honest rows, so test never leaks into it. RBAC/ABAC need no retraining, so -- unlike the ML baseline -- every variant is evaluated on its OWN sessions, not a model trained elsewhere.

In [ ]:
import os
import pandas as pd
import benchmark_labels, temporal_split, naive_trust, rbac_baseline, abac_baseline
from decision_engine import Decision

COMBINED_CSV = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
OUT_DIR = os.environ.get('V2G_RBAC_ABAC_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds')
MALICIOUS_CPO = 'CPO-B'

df = pd.read_csv(COMBINED_CSV)
df['connection_time'] = pd.to_datetime(df['connection_time'], utc=True)
df = benchmark_labels.add_ground_truth_labels(df)

day_cutoff = temporal_split.compute_day_cutoff(df)
train_mask = df['day_idx'] < day_cutoff
cpo_trust_scale = naive_trust.compute_cpo_trust_scale(df[train_mask], MALICIOUS_CPO, per_variant=True)
print(f'day_cutoff={day_cutoff}')
print(f'cpo_trust_scale (train-only, per variant): {cpo_trust_scale}')

df = naive_trust.add_all_naive_trust(df, cpo_trust_scale=cpo_trust_scale)
df = rbac_baseline.add_rbac_decision(df)
df = abac_baseline.add_abac_decisions(df)

variants = sorted(df['variant'].unique()) if 'variant' in df.columns else [None]
summary_rows = []
for variant in variants:
    vdf = df if variant is None else df[df['variant'] == variant]
    print(f'\n{"="*70}\n=== variant={variant} ===\n{"="*70}')
    print('\n--- RBAC: structural blindness ---')
    rbac_baseline.summarize_rbac_blindness(vdf)
    print('\n--- ABAC: single-authority vs multi-authority ---')
    abac_baseline.summarize_single_vs_multi(vdf)

    risky = vdf[vdf['risk_factor_count'] > 0]
    rbac_permit_rate = (risky['rbac_decision'] == Decision.PERMIT.value).mean() if len(risky) else float('nan')
    blind_spot = vdf[(vdf['gt_cpo_malicious'] == 0) & ((vdf['gt_emsp_malicious'] == 1) | (vdf['gt_ev_sybil'] == 1))]
    not_denied_single = (blind_spot['abac_single_decision'] != Decision.DENY.value).mean() if len(blind_spot) else float('nan')
    denied_multi = (blind_spot['abac_multi_decision'] == Decision.DENY.value).mean() if len(blind_spot) else float('nan')
    summary_rows.append({'variant': variant, 'n_sessions': len(vdf), 'n_risky': len(risky),
                          'rbac_permit_rate_on_risky': rbac_permit_rate, 'n_blind_spot': len(blind_spot),
                          'abac_single_not_denied_pct': 100 * not_denied_single,
                          'abac_multi_denied_pct': 100 * denied_multi})

summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUT_DIR, 'rbac_abac_variant_summary.csv')
summary.to_csv(summary_path, index=False)
print(f'\n{"="*70}\n=== Per-variant summary (saved to {summary_path}) ===')
print(summary.round(2).to_string(index=False))

seed_variants = [v for v in variants if v and v.startswith('seed_')]
if seed_variants:
    stab = summary[summary['variant'].isin(seed_variants)]
    print(f'\nStability across {len(seed_variants)} reseeds -- abac_single_not_denied_pct '
          f'mean={stab["abac_single_not_denied_pct"].mean():.2f}  '
          f'std={stab["abac_single_not_denied_pct"].std():.2f}')

out_path = os.path.join(OUT_DIR, 'v2g_rbac_abac_decisions.csv')
df.to_csv(out_path, index=False)
print(f'\nSaved full decisioned dataset -> {out_path}')

## Step 4: insights -- per-variant blind-spot chart

One chart, reusing `rbac_abac_variant_summary.csv` from Step 3: for every variant, how often single-authority ABAC fails to deny a genuine blind-spot session (honest-looking CPO, malicious eMSP or Sybil EV) versus how often multi-authority ABAC catches the SAME sessions. `adversarial` is the one build expected to move this the most (camouflaged CPO fraud is the hardest case); `sparse_feedback`/`noisy_benign` are expected to move it only a little, by design -- see `multi_world_pipeline.py`'s `VARIANTS` dict and the reference ACCESS-IoT paper's own Table 5 pattern, which this mirrors.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = os.environ.get('V2G_RBAC_ABAC_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds')
summary = pd.read_csv(os.path.join(OUT_DIR, 'rbac_abac_variant_summary.csv'))
summary = summary.set_index('variant')

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(summary))
width = 0.35
ax.bar([i - width/2 for i in x], summary['abac_single_not_denied_pct'], width, label='Single-authority: NOT denied (%)', color='#d9534f')
ax.bar([i + width/2 for i in x], summary['abac_multi_denied_pct'], width, label='Multi-authority: denied (%)', color='#5cb85c')
ax.set_xticks(list(x))
ax.set_xticklabels(summary.index, rotation=30, ha='right')
ax.set_ylabel('% of blind-spot sessions')
ax.set_title('ABAC blind-spot: single- vs multi-authority, per variant')
ax.set_ylim(0, 115)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('rbac_abac_variant_insights.pdf', dpi=300)
plt.show()
print('Saved rbac_abac_variant_insights.png')

## Done

`v2g_rbac_abac_decisions.csv` (every session's RBAC decision plus both ABAC variants' decisions, alongside the naive trust scores they were computed from) and `rbac_abac_variant_summary.csv` (the per-variant table) in the same `worlds/` folder. Paste back the printed per-variant summary table and the saved chart.